# Notebook 05 — Missing Frame Experiment

## Research Question
> *When frames are deliberately dropped from a tracking sequence, which pipeline — Classical ML or Deep Learning — better preserves person identity across the gap?*

## Experiment Design

### Drop Scenarios
| Scenario | Description | Frames Dropped |
|---|---|---|
| Baseline | No drops | 0% |
| Random 5% | Random frames removed | ~5% |
| Random 10% | Random frames removed | ~10% |
| Random 20% | Random frames removed | ~20% |
| Burst drop | Consecutive block removed | 10 consecutive frames × N points |

### Why Burst Drops?
Real surveillance systems often fail in **bursts** — network packets lost, camera occlusion, frame decoder errors. Burst drops create harder re-identification challenges than random drops because the tracker must bridge a longer continuous gap.

### Evaluation
The tracker receives ONLY the non-dropped frames. After tracking, we evaluate:
- **MOTA** = $1 - \frac{\text{FP} + \text{FN} + \text{IDSW}}{\text{GT}}$
- **MOTP** = $\frac{\sum_{t,i} d(i,t)}{\sum_{t,i} \text{matched}_{t,i}}$
- **IDF1** = $\frac{2 \cdot \text{IDTP}}{2 \cdot \text{IDTP} + \text{IDFP} + \text{IDFN}}$
- **IDSW** = number of identity switches

In [ ]:
import os, json, pickle, cv2, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
import torch, xgboost as xgb
import torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import StandardScaler
from PIL import Image
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

with open(os.path.join(
    r'D:\MTech\Sem-2\IT585-Advanced_ML\AML-project\VisDrone',
    'VisDrone_outputs', 'config.json')) as f:
    cfg = json.load(f)

TRAIN_DIR=cfg['TRAIN_DIR']; VAL_DIR=cfg['VAL_DIR']
OUTPUT_DIR=cfg['OUTPUT_DIR']; VALID_CLASSES=set(cfg['VALID_CLASSES'])
VAL_SEQS=cfg['VAL_SEQS']
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
print(f'Val sequences available: {len(VAL_SEQS)}')

## Cell 1 — Frame Drop Generator

In [ ]:
# ──────────────────────────────────────────────────────────────────
# FRAME DROP GENERATORS
# Create different patterns of missing frames to stress-test
# the trackers' re-identification capability.
# ──────────────────────────────────────────────────────────────────

def generate_random_drops(frame_ids, drop_rate, seed=42):
    """
    Randomly drop a fraction of frame IDs.

    Args:
        frame_ids: sorted list of all frame IDs in sequence
        drop_rate: fraction to drop (0.05 = 5%)
        seed     : random seed for reproducibility

    Returns:
        set of frame IDs to drop
    """
    rng = random.Random(seed)
    n_drop = int(len(frame_ids) * drop_rate)
    dropped = set(rng.sample(frame_ids, n_drop))
    print(f'Random drop {drop_rate*100:.0f}%: {len(dropped)} frames dropped '
          f'out of {len(frame_ids)} total')
    return dropped


def generate_burst_drops(frame_ids, burst_length=10, n_bursts=None, seed=42):
    """
    Drop consecutive blocks of frames (burst drops).
    Simulates camera outage or network failure.

    Args:
        frame_ids   : sorted list of all frame IDs
        burst_length: number of consecutive frames per burst
        n_bursts    : number of burst locations (default = len//50)
        seed        : random seed

    Returns:
        set of frame IDs to drop
    """
    rng = random.Random(seed)
    if n_bursts is None:
        n_bursts = max(1, len(frame_ids) // 50)

    # Pick burst start positions (not too close to edges)
    margin = burst_length * 2
    valid_starts = frame_ids[margin:-margin-burst_length]
    if len(valid_starts) < n_bursts:
        valid_starts = frame_ids[:-burst_length]

    start_positions = rng.sample(valid_starts, min(n_bursts, len(valid_starts)))

    # Build set of dropped frame IDs
    frame_set = set(frame_ids)
    dropped = set()
    for start in start_positions:
        idx = frame_ids.index(start)
        for i in range(burst_length):
            if idx + i < len(frame_ids):
                dropped.add(frame_ids[idx + i])

    print(f'Burst drop: {n_bursts} bursts × {burst_length} frames = '
          f'{len(dropped)} frames dropped')
    return dropped


# Test on first val sequence
def read_visdrone_annotation(anno_path, valid_classes=None):
    cols=['frame_id','target_id','x','y','w','h',
          'score','class_id','truncation','occlusion']
    df=pd.read_csv(anno_path,header=None,names=cols)
    df=df[df['score']==1]; df=df[df['target_id']>0]
    if valid_classes: df=df[df['class_id'].isin(valid_classes)]
    return df[(df['w']>0)&(df['h']>0)].reset_index(drop=True)


test_seq   = VAL_SEQS[0]
anno_path  = os.path.join(VAL_DIR, 'annotations', test_seq + '.txt')
df_test    = read_visdrone_annotation(anno_path, VALID_CLASSES)
frame_ids  = sorted(df_test['frame_id'].unique().tolist())
print(f'Sequence: {test_seq}')
print(f'Total frames: {len(frame_ids)}')
print()

# Generate all drop scenarios
drop_scenarios = {
    'baseline' : set(),
    'random_5' : generate_random_drops(frame_ids, 0.05),
    'random_10': generate_random_drops(frame_ids, 0.10),
    'random_20': generate_random_drops(frame_ids, 0.20),
    'burst'    : generate_burst_drops(frame_ids)
}

# Save drop scenarios
drops_dir = os.path.join(OUTPUT_DIR, 'dropped_frames')
os.makedirs(drops_dir, exist_ok=True)
for name, drops in drop_scenarios.items():
    path = os.path.join(drops_dir, f'{test_seq}_{name}.txt')
    with open(path, 'w') as f:
        f.write('\n'.join(map(str, sorted(drops))))
print('\n✅ Drop scenarios saved!')

## Cell 2 — Re-Import All Classes and Run Both Trackers for All Scenarios

In [ ]:
# ── Re-import classical feature extractor ─────────────────────────
def extract_hsv_histogram(bgr_crop, bins=(16,16,8)):
    hsv=cv2.cvtColor(bgr_crop,cv2.COLOR_BGR2HSV)
    hist=cv2.calcHist([hsv],[0,1,2],None,list(bins),[0,180,0,256,0,256])
    hist=hist.flatten().astype(np.float32)
    total=hist.sum()
    if total>0: hist/=total
    return hist

def extract_lbp_histogram(bgr_crop,points=8,radius=1):
    gray=cv2.cvtColor(bgr_crop,cv2.COLOR_BGR2GRAY)
    lbp=local_binary_pattern(gray,points,radius,method='uniform')
    hist,_=np.histogram(lbp,bins=points+2,range=(0,points+2))
    hist=hist.astype(np.float32)
    total=hist.sum()
    if total>0: hist/=total
    return hist

def extract_classical_feature(bgr_crop):
    return np.concatenate([extract_hsv_histogram(bgr_crop),
                           extract_lbp_histogram(bgr_crop)])

def compute_pair_feature(f1,f2,eps=1e-8):
    abs_diff=np.abs(f1-f2)
    n1=np.linalg.norm(f1)+eps; n2=np.linalg.norm(f2)+eps
    cos_dist=np.array([1.0-np.dot(f1,f2)/(n1*n2)])
    chi2=np.array([np.sum((f1-f2)**2/(f1+f2+eps))])
    return np.concatenate([abs_diff,cos_dist,chi2])

def compute_iou_matrix(bboxes_a,bboxes_b):
    N,M=len(bboxes_a),len(bboxes_b); iou=np.zeros((N,M))
    for i,a in enumerate(bboxes_a):
        for j,b in enumerate(bboxes_b):
            xi1=max(a[0],b[0]);yi1=max(a[1],b[1])
            xi2=min(a[2],b[2]);yi2=min(a[3],b[3])
            inter=max(0,xi2-xi1)*max(0,yi2-yi1)
            area_a=(a[2]-a[0])*(a[3]-a[1])
            area_b=(b[2]-b[0])*(b[3]-b[1])
            iou[i,j]=inter/(area_a+area_b-inter+1e-8)
    return iou

# Load trained XGBoost + scaler
xgb_model=xgb.XGBClassifier()
xgb_model.load_model(os.path.join(OUTPUT_DIR,'models','xgb_reid.json'))
with open(os.path.join(OUTPUT_DIR,'models','classical_scaler.pkl'),'rb') as f:
    scaler=pickle.load(f)

# ── Re-import deep feature extractor ──────────────────────────────
class DeepFeatureExtractor(nn.Module):
    def __init__(self,device):
        super().__init__()
        backbone=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone=nn.Sequential(*list(backbone.children())[:-2])
        self.gap=nn.AdaptiveAvgPool2d(1)
        self.device=device; self.to(device); self.eval()
    @torch.no_grad()
    def forward(self,x):
        f=self.backbone(x); f=self.gap(f).view(x.size(0),-1)
        return F.normalize(f,p=2,dim=1)
    def extract_batch(self,bgr_crops):
        if not bgr_crops: return np.empty((0,2048),dtype=np.float32)
        transform=T.Compose([T.Resize((256,128)),T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        tensors=[transform(Image.fromarray(cv2.cvtColor(c,cv2.COLOR_BGR2RGB)))
                 for c in bgr_crops]
        batch=torch.stack(tensors).to(self.device)
        return self.forward(batch).cpu().numpy()

deep_extractor=DeepFeatureExtractor(DEVICE)
print('All components loaded!')
print('Running all 5 scenarios × 2 pipelines...')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# RUN BOTH PIPELINES FOR ALL DROP SCENARIOS
# Results are saved to disk and also stored in a dict
# for evaluation in Notebook 06.
# ──────────────────────────────────────────────────────────────────

# (Kalman + ClassicalSORT + DeepSORT class definitions omitted here
# for brevity — paste them from notebooks 03 and 04 before running)
# --- OR --- run notebooks 03 and 04 first to save models, then run
# the tracking functions from those notebooks by importing them.

# For a self-contained run, copy the class definitions here.
# The full classes are defined in 03_classical_pipeline.ipynb
# and 04_deep_pipeline.ipynb.

all_results = {}  # {(pipeline, scenario): track_results_list}
seq = VAL_SEQS[0]

for scenario_name, dropped in drop_scenarios.items():
    print(f'\n--- Scenario: {scenario_name} | Dropped: {len(dropped)} frames ---')

    # ── Classical pipeline ─────────────────────────────────────────
    # Uncomment once ClassicalSORT is defined (copy from nb 03)
    # classical_results = run_classical_tracker_on_sequence(
    #     VAL_DIR, seq, xgb_model, scaler, VALID_CLASSES,
    #     dropped_frames=dropped,
    #     save_dir=os.path.join(OUTPUT_DIR, 'tracks', 'classical')
    # )
    # all_results[('classical', scenario_name)] = classical_results
    # print(f'  Classical: {len(classical_results):,} track outputs')

    # ── Deep pipeline ──────────────────────────────────────────────
    # Uncomment once DeepSORT is defined (copy from nb 04)
    # deep_results = run_deep_tracker_on_sequence(
    #     VAL_DIR, seq, deep_extractor, VALID_CLASSES,
    #     dropped_frames=dropped,
    #     save_dir=os.path.join(OUTPUT_DIR, 'tracks', 'deep')
    # )
    # all_results[('deep', scenario_name)] = deep_results
    # print(f'  Deep     : {len(deep_results):,} track outputs')

    print(f'  [Copy class definitions from NB03/NB04 to enable tracking here]')

# Save results dict
import pickle
with open(os.path.join(OUTPUT_DIR, 'all_track_results.pkl'), 'wb') as f:
    pickle.dump(all_results, f)

print('\n✅ All scenarios complete!')
print('Next: Notebook 06 — Evaluation and Comparison')